<a href="https://colab.research.google.com/github/Quentalh/Flyrank_Heitor_Quental_ML_Track/blob/main/work/notebooks/Copy_of_02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [37]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [38]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [39]:
# @title
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.950
Hand rule  Precision@50: 0.660


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [40]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [41]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.950   vs   tree 0.650
Precision@50:  hand rule 0.660   vs   tree 0.640


Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [42]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [43]:
# Your experiment here
from sklearn.model_selection import train_test_split

#Method for splitting data from script 3
def make_client_aware_split(
    frame: pd.DataFrame,
    target_series: pd.Series,
) -> tuple[np.ndarray, np.ndarray, str]:
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()

    if len(unique_clients) >= 5:
        random_generator = np.random.default_rng(42)
        shuffled_clients = random_generator.permutation(unique_clients)
        test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
        test_clients = set(shuffled_clients[:test_client_count])
        test_mask = client_series.isin(test_clients).to_numpy()
        train_indices = all_indices[~test_mask]
        test_indices = all_indices[test_mask]

        if (
            len(train_indices) > 0
            and len(test_indices) > 0
            and target_series.iloc[train_indices].nunique() == 2
            and target_series.iloc[test_indices].nunique() == 2
        ):
            return train_indices, test_indices, "client_holdout"

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.2,
        random_state=42,
        stratify=target_series,
    )
    return np.array(train_indices), np.array(test_indices), "stratified_row_holdout"

#The rest of this block is the experiment which changes the max depth from 2 to 3, and then later 4

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X, y)
tree2 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree2.fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]
tree_score2 = tree2.predict_proba(X)[:, 1]
k_para_3 = precision_at_k(tree_score, y, 50)
k_para_4 = precision_at_k(tree_score2, y, 50)
print(f"Precision@50: {k_para_3:.3f}\n")

print(f'{export_text(tree, feature_names=features)}\n')

print(f"Precision@50: {k_para_4:.3f}\n")

print(f'{export_text(tree2, feature_names=features)}\n')

print("The precisison seems to improve with the amount of depth added to the decision tree,  but the tree also becomes \n increasingly difficult to read, a fact that becomes immediately noticeable, in the context of this experiment \n when max_depth is set to 4.")




Precision@50: 0.660

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0


Precision@50: 0.720

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--

In [44]:
#This block contains the experiment which changes the set of features, removing impressions_90d and adding engagement_rate

# Defining the new set of features
features_modified = ["content_age_days", "days_since_last_update",
                     "avg_position", "ctr", "word_count", "engagement_rate"]

X_modified_in_sample = df[features_modified].replace([np.inf, -np.inf], np.nan).fillna(0)
tree_modified_in_sample = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_modified_in_sample.fit(X_modified_in_sample, y)
tree_score_modified_in_sample = tree_modified_in_sample.predict_proba(X_modified_in_sample)[:, 1]
precision_modified_in_sample = precision_at_k(tree_score_modified_in_sample, y, 50)

print(f"Precision@50 (new set of features): {precision_modified_in_sample:.3f}\n")
print('Decision Tree (new set of features): ')
print(export_text(tree_modified_in_sample, feature_names=features_modified))

print("The tree seems to have chosen the avg_position feature to split on first")

Precision@50 (new set of features): 0.660

Decision Tree (new set of features): 
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

The tree seems to have chosen the avg_position feature to split on first


In [45]:
#Block for performing the data split using make_client_aware_split

features_for_split = features
X_data_for_split = df[features_for_split].replace([np.inf, -np.inf], np.nan).fillna(0)
y_data_for_split = df["is_declining_label"]
train_indices, test_indices, split_type = make_client_aware_split(df, y_data_for_split)
X_train = X_data_for_split.iloc[train_indices]
X_test = X_data_for_split.iloc[test_indices]
y_train = y_data_for_split.iloc[train_indices].values
y_test = y_data_for_split.iloc[test_indices].values

print(f"Split Type: {split_type}")
print(f"Train set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")
print(f"Test set declining rate: {y_test.mean():.3f}")

print("With the data having been split we can see that declining rate seems to have lowered")

Split Type: client_holdout
Train set size: 27675 samples
Test set size: 2325 samples
Test set declining rate: 0.391
With the data having been split we can see that declining rate seems to have lowered


In [46]:
from sklearn.tree import DecisionTreeClassifier, export_text

# Decision tree with max_depth=3, trained on split training data
tree_depth_3_split = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_depth_3_split.fit(X_train, y_train)
tree_score_3_test = tree_depth_3_split.predict_proba(X_test)[:, 1]
k_para_3_test = precision_at_k(tree_score_3_test, y_test, 50)
print(f"Precision@50 (max_depth=3, Test Set): {k_para_3_test:.3f}\n")
print(export_text(tree_depth_3_split, feature_names=features))

print("\n" +"\n")

# Decision tree with max_depth=4, trained on split training data
tree_depth_4_split = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_depth_4_split.fit(X_train, y_train)
tree_score_4_test = tree_depth_4_split.predict_proba(X_test)[:, 1]
k_para_4_test = precision_at_k(tree_score_4_test, y_test, 50)
print(f"Precision@50 (max_depth=4, Test Set): {k_para_4_test:.3f}\n")
print(export_text(tree_depth_4_split, feature_names=features))

print("THe precision from the trees with 3 or 4 as the max_depth numbers, seems to be the same in both cases, and it also is lower than the scores obtained when using the in-sample data pipeline,\n which means it is more protected against overfitting and is more generalized")

Precision@50 (max_depth=3, Test Set): 0.580

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.90
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.90
|   |   |--- content_age_days <= 97.00
|   |   |   |--- class: 0
|   |   |--- content_age_days >  97.00
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0




Precision@50 (max_depth=4, Test Set): 0.580

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.90
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- days_since_last_update <= 56.00
|   |   |   |   |--- class: 0
|   |   |   |--- days_since_last_update >  56.00
|   |   |   

In [47]:
 #Same modified set of features
features_modified_split = ["content_age_days", "days_since_last_update",
                          "avg_position", "ctr", "word_count", "engagement_rate"]

# Prepare the data with new features for splitting
X_data_for_split_modified = df[features_modified_split].replace([np.inf, -np.inf], np.nan).fillna(0)
train_indices_modified, test_indices_modified, split_type_modified = make_client_aware_split(df, y_data_for_split)

# Create train/test sets using the new features and split indices
X_train_modified = X_data_for_split_modified.iloc[train_indices_modified]
X_test_modified = X_data_for_split_modified.iloc[test_indices_modified]
y_train_modified = y_data_for_split.iloc[train_indices_modified].values
y_test_modified = y_data_for_split.iloc[test_indices_modified].values

print(f"Split Type for modified features: {split_type_modified}")
print(f"Train set size (modified features): {len(X_train_modified)} samples")
print(f"Test set size (modified features): {len(X_test_modified)} samples")
print(f"Test set declining rate (modified features): {y_test_modified.mean():.3f}\n")

# Train a decision tree (max_depth=2) on the split training data with new features
tree_modified_split = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_modified_split.fit(X_train_modified, y_train_modified)

# Predict probabilities on the split test set
tree_score_modified_test = tree_modified_split.predict_proba(X_test_modified)[:, 1]

# Recalculate hand rule score for the test set using the modified test_indices
stale_test_modified = (df.iloc[test_indices_modified]["days_since_last_update"] >= 180).astype(int)
visible_test_modified = (df.iloc[test_indices_modified]["impressions_90d"] >= 500).astype(int)
hand_rule_score_test_modified = stale_test_modified * visible_test_modified * df.iloc[test_indices_modified]["impressions_90d"]

print("--- Split Test Set Results (new set of features) ---")

for k in (20, 50):
    hr_test_modified = precision_at_k(hand_rule_score_test_modified, y_test_modified, k)
    tr_test_modified = precision_at_k(tree_score_modified_test, y_test_modified, k)
    print(f"Precision@{k} (Test Set):  Hand rule {hr_test_modified:.3f}   vs   Tree {tr_test_modified:.3f}")

print("Decision Tree Structure (Client-Aware Split, new features):")
print(export_text(tree_modified_split, feature_names=features_modified_split))

print("The decision Tree model obtained better results than the hand rule when k = 50, but not when k = 20.")

Split Type for modified features: client_holdout
Train set size (modified features): 27675 samples
Test set size (modified features): 2325 samples
Test set declining rate (modified features): 0.391

--- Split Test Set Results (new set of features) ---
Precision@20 (Test Set):  Hand rule 0.600   vs   Tree 0.500
Precision@50 (Test Set):  Hand rule 0.460   vs   Tree 0.480
Decision Tree Structure (Client-Aware Split, new features):
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 310.50
|   |   |--- class: 1
|   |--- content_age_days >  310.50
|   |   |--- class: 0

The decision Tree model obtained better results than the hand rule when k = 50, but not when k = 20.


In [48]:
#This Block is only to show directly the comparison between the results from the tests, one from the pipeline that uses In-sample data, and the other from the pipeline that uses Split data.

tree_split_client_aware = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_split_client_aware.fit(X_train, y_train)
tree_score_test_client_aware = tree_split_client_aware.predict_proba(X_test)[:, 1]
stale_test_client_aware = (df.iloc[test_indices]["days_since_last_update"] >= 180).astype(int)
visible_test_client_aware = (df.iloc[test_indices]["impressions_90d"] >= 500).astype(int)
hand_rule_score_test_client_aware = stale_test_client_aware * visible_test_client_aware * df.iloc[test_indices]["impressions_90d"]

print("Client-Aware Split Test Set Results")
for k in (20, 50):
    hr_test_client_aware = precision_at_k(hand_rule_score_test_client_aware, y_test, k)
    tr_test_client_aware = precision_at_k(tree_score_test_client_aware, y_test, k)
    print(f"Precision@{k} (Test Set):  Hand rule {hr_test_client_aware:.3f}   vs   Tree {tr_test_client_aware:.3f}")

print("\nOriginal In-Sample Results (for comparison)")
for k in (20, 50):
    hr_original = precision_at_k(df["hand_rule_score"], y, k)
    tr_original = precision_at_k(tree.predict_proba(X)[:, 1], y, k)
    print(f"Precision@{k} (In-Sample): Hand rule {hr_original:.3f}   vs   Tree {tr_original:.3f}")

Client-Aware Split Test Set Results
Precision@20 (Test Set):  Hand rule 0.600   vs   Tree 0.400
Precision@50 (Test Set):  Hand rule 0.460   vs   Tree 0.440

Original In-Sample Results (for comparison)
Precision@20 (In-Sample): Hand rule 0.950   vs   Tree 0.650
Precision@50 (In-Sample): Hand rule 0.660   vs   Tree 0.660


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.